# Table of Contents
* [Section 1: Imports & Configuration](#imports)
* [Section 2: Extracting Words With Their Timecodes](#extract-words)
* [Section 3: Mapping Words To Phonemes](#map-words)
* [Section 4: Rhyme Unit Extraction](#extract-rhyme-units)
* [Section 5: Pairwise Rhyme Similarity](#pairwise-similarity)
* [Section 6: Words Clustering](#clustering)
* [Section 7: HTML Generation](#html-generation)

---
## Section 1: Imports & Configuration <a id="imports"></a>
---

In [ ]:
import sys
sys.path.append("..")
print(sys.executable)

from python.rhyme_extraction import (
    extract_rhyme_unit, 
    extract_end_words, 
    extract_rhyme_candidates
)

from python.similarity_engine import (
    rhyme_similarity,
    compute_similarity_pairs,
    build_similarity_matrix
)
from python.clustering import (
    SIMILARITY_THRESHOLD,
    get_threshold,
    cluster_rhymes, 
    cluster_exact_rhymes
)
from python.debugging import debug_rhyme_analysis
from python.evaluation import evaluate_rhyme_detection
from python.html_generation import generate_rhyme_html

from praatio import textgrid

RHYME_MODE = "stressed"
DETECTION_MODE = "full_line"
# Options for RHYME_MODE:   "stressed" / "stressed_plus" / "entire_word"
# Options for DETECTION_MODE: "end_only" / "full_line"

INPUT_TXT_PATH = "input/current_input/input.txt"
OUTPUT_TEXTGRID_PATH = "output/current_output/input.TextGrid"

---
## Section 2: Extracting Words With Their Timecodes <a id="extract-words"></a>
---

In [2]:
tg = textgrid.openTextgrid(OUTPUT_TEXTGRID_PATH, includeEmptyIntervals=False)

word_tier = tg.getTier("words")
phone_tier = tg.getTier("phones")

for word in word_tier.entries:
    print(word)

Interval(start=1.68, end=1.82, label='i')
Interval(start=1.82, end=2.24, label='found')
Interval(start=2.24, end=2.3, label='the')
Interval(start=2.3, end=2.81, label='sound')
Interval(start=2.93, end=3.1, label='of')
Interval(start=3.1, end=3.21, label='the')
Interval(start=3.21, end=4.03, label='underground')
Interval(start=4.34, end=4.43, label='the')
Interval(start=4.43, end=4.89, label='crowd')
Interval(start=4.89, end=5.14, label='was')
Interval(start=5.14, end=5.57, label='loud')
Interval(start=5.62, end=5.84, label='and')
Interval(start=5.84, end=6.33, label='proud')
Interval(start=6.33, end=6.85, label='around')
Interval(start=6.9, end=7.0, label='the')
Interval(start=7.0, end=7.55, label='mound')
Interval(start=8.08, end=8.17, label='a')
Interval(start=8.17, end=8.58, label='light')
Interval(start=8.66, end=8.86, label='at')
Interval(start=8.86, end=9.24, label='night')
Interval(start=9.24, end=10.04, label='ignites')
Interval(start=10.15, end=10.31, label='the')
Interval(sta

---
## Section 3: Mapping Words To Phonemes <a id="map-words"></a>
---

In [3]:
for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    phonemes = []
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    print(word_label + ":", phonemes)

# Populating the word_to_phonemes dictionary
word_to_phonemes = {}

for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    
    phonemes = []
    
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    
    word_label = word_label.lower()
    
    if word_label not in word_to_phonemes:
        word_to_phonemes[word_label] = phonemes

print(word_to_phonemes)

i: ['AY1']
found: ['F', 'AW1', 'N', 'D']
the: ['DH', 'AH0']
sound: ['S', 'AW1', 'N', 'D']
of: ['AH1', 'V']
the: ['DH', 'IY0']
underground: ['AH1', 'N', 'D', 'ER0', 'G', 'R', 'AW2', 'N', 'D']
the: ['DH', 'AH0']
crowd: ['K', 'R', 'AW1', 'D']
was: ['W', 'AH0', 'Z']
loud: ['L', 'AW1', 'D']
and: ['AE1', 'N', 'D']
proud: ['P', 'R', 'AW1', 'D']
around: ['ER0', 'AW1', 'N', 'D']
the: ['DH', 'AH0']
mound: ['M', 'AW1', 'N', 'D']
a: ['AH0']
light: ['L', 'AY1', 'T']
at: ['AE1', 'T']
night: ['N', 'AY1', 'T']
ignites: ['IH0', 'G', 'N', 'AY1', 'T', 'S']
the: ['DH', 'AH0']
kite: ['K', 'AY1', 'T']
in: ['IH0', 'N']
flight: ['F', 'L', 'AY1', 'T']
the: ['DH', 'AH0']
sight: ['S', 'AY1', 'T']
of: ['AH0', 'V']
white: ['W', 'AY1', 'T']
delight: ['D', 'IH0', 'L', 'AY1', 'T']
shines: ['SH', 'AY1', 'N', 'Z']
bright: ['B', 'R', 'AY1', 'T']
tonight: ['T', 'AH0', 'N', 'AY1', 'T']
{'i': ['AY1'], 'found': ['F', 'AW1', 'N', 'D'], 'the': ['DH', 'AH0'], 'sound': ['S', 'AW1', 'N', 'D'], 'of': ['AH1', 'V'], 'underground': 

---
## Section 4: Rhyme Unit Extraction <a id="extract-rhyme-units"></a>
---

In [4]:
rhyme_candidates = extract_rhyme_candidates(
    INPUT_TXT_PATH,
    word_to_phonemes,
    rhyme_mode=RHYME_MODE,
    detection_mode=DETECTION_MODE
)

for r in rhyme_candidates:
    print(r)

{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 0, 'end_word': 'i', 'phonemes': ['AY1'], 'rhyme_unit': ['AY1'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 1, 'end_word': 'found', 'phonemes': ['F', 'AW1', 'N', 'D'], 'rhyme_unit': ['AW1', 'N', 'D'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 2, 'end_word': 'the', 'phonemes': ['DH', 'AH0'], 'rhyme_unit': ['AH0'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 3, 'end_word': 'sound', 'phonemes': ['S', 'AW1', 'N', 'D'], 'rhyme_unit': ['AW1', 'N', 'D'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 4, 'end_word': 'of', 'phonemes': ['AH1', 'V'], 'rhyme_unit': ['AH1', 'V'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'I found the sound of the underground

---
## Section 5: Pairwise Rhyme Similarity <a id="pairwise-similarity"></a>
---

In [5]:
similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)

for pair in similarity_pairs:
    print(pair)

# Building a similarity matrix
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)

for w1 in similarity_matrix:
    print(w1, similarity_matrix[w1])

{'wordA': 'i', 'wordB': 'found', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'sound', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'of', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'underground', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'crowd', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'was', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'loud', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'and', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'proud', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'around', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'the', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'mound', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'a', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'light', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'at', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'night', 'similarity': 0.0}
{'wordA': 'i', 'wordB': 'ign

In [11]:
# Milestone 9 — slant rhyme spot-check
from python.similarity_engine import phoneme_similarity, longest_common_tail_similarity

spot_check_pairs = [
    # Perfect rhymes — expect 1.0
    ("light",   ["AY1", "T"],           "night",   ["AY1", "T"]),
    ("found",   ["AW1", "N", "D"],      "mound",   ["AW1", "N", "D"]),
    ("proud",   ['AW1', 'D'],           "mound",   ["AW1", "N", "D"]),

    # Slant rhymes (same class) — expect > 0.7
    ("late",    ["AY1", "T"],           "made",    ["AY1", "D"]),       # T/D same class
    ("face",    ["AY1", "S"],           "days",    ["AY1", "Z"]),       # S/Z same class

    # Near-vowel slant — expect ~0.4–0.7
    ("beat",    ["IY1", "T"],           "bit",     ["IH1", "T"]),       # IY/IH same class
    ("fate",    ["EY1", "T"],           "feet",    ["IY1", "T"]),       # EY/IY same superclass

    # Genuinely different — expect low / 0.0
    ("light",   ["AY1", "T"],           "found",   ["AW1", "N", "D"]),
    ("the",     ["DH", "AH0"],          "mound",   ["AW1", "N", "D"]),
]

print(f"{'Word A':<10} {'Word B':<10} {'Score':>6}  Rhyme units")
print("-" * 55)
for wordA, ruA, wordB, ruB in spot_check_pairs:
    score = longest_common_tail_similarity(ruA, ruB)
    print(f"{wordA:<10} {wordB:<10} {score:>6.3f}  {ruA} | {ruB}")

Word A     Word B      Score  Rhyme units
-------------------------------------------------------
light      night       1.000  ['AY1', 'T'] | ['AY1', 'T']
found      mound       1.000  ['AW1', 'N', 'D'] | ['AW1', 'N', 'D']
proud      mound       0.500  ['AW1', 'D'] | ['AW1', 'N', 'D']
late       made        0.850  ['AY1', 'T'] | ['AY1', 'D']
face       days        0.850  ['AY1', 'S'] | ['AY1', 'Z']
beat       bit         0.850  ['IY1', 'T'] | ['IH1', 'T']
fate       feet        0.700  ['EY1', 'T'] | ['IY1', 'T']
light      found       0.350  ['AY1', 'T'] | ['AW1', 'N', 'D']
the        mound       0.000  ['DH', 'AH0'] | ['AW1', 'N', 'D']


---
## Section 6: Words Clustering <a id="clustering"></a>
---

In [7]:
# Clustering by thresholds from the similarity matrix
# get_threshold() returns the correct value for the active RHYME_MODE
cluster_labels, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

print("Clusters:")
print(clusters)

print("\nWord → Cluster Mapping:")
print(cluster_labels)

# Defining expected_clusters and comparing with the currect clustering
expected_clusters = {
    "underground": "A",
    "mound": "A",
    "flight": "B",
    "tonight": "B"
}

for r in rhyme_candidates:
    word = r["end_word"]
    r["expected_cluster"] = expected_clusters.get(word, None)

for r in rhyme_candidates:
    print(r)

# Exact rhymes clusters printout
clusters = cluster_labels

print(clusters)

# Debugging the clustering
debug_rhyme_analysis(rhyme_candidates, clusters)

# Accuracy evaluation
accuracy, mismatches = evaluate_rhyme_detection(rhyme_candidates, clusters)

Clusters:
{'A': ['i'], 'B': ['found', 'sound', 'underground', 'around', 'mound'], 'C': ['the', 'a'], 'D': ['of', 'was'], 'E': ['crowd', 'loud', 'proud'], 'F': ['and'], 'G': ['light', 'night', 'kite', 'flight', 'sight', 'white', 'delight', 'bright', 'tonight'], 'H': ['at'], 'I': ['ignites'], 'J': ['in'], 'K': ['shines']}

Word → Cluster Mapping:
{'i': 'A', 'found': 'B', 'sound': 'B', 'underground': 'B', 'around': 'B', 'mound': 'B', 'the': 'C', 'a': 'C', 'of': 'D', 'was': 'D', 'crowd': 'E', 'loud': 'E', 'proud': 'E', 'and': 'F', 'light': 'G', 'night': 'G', 'kite': 'G', 'flight': 'G', 'sight': 'G', 'white': 'G', 'delight': 'G', 'bright': 'G', 'tonight': 'G', 'at': 'H', 'ignites': 'I', 'in': 'J', 'shines': 'K'}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 0, 'end_word': 'i', 'phonemes': ['AY1'], 'rhyme_unit': ['AY1'], 'is_line_end': False, 'expected_cluster': None}
{'line_index': 0, 'line_text': 'I found the sound of the underground', 'word_index': 1

In [8]:
print(clusters)

{'i': 'A', 'found': 'B', 'sound': 'B', 'underground': 'B', 'around': 'B', 'mound': 'B', 'the': 'C', 'a': 'C', 'of': 'D', 'was': 'D', 'crowd': 'E', 'loud': 'E', 'proud': 'E', 'and': 'F', 'light': 'G', 'night': 'G', 'kite': 'G', 'flight': 'G', 'sight': 'G', 'white': 'G', 'delight': 'G', 'bright': 'G', 'tonight': 'G', 'at': 'H', 'ignites': 'I', 'in': 'J', 'shines': 'K'}


In [9]:
# Milestone 9 — full pipeline regression check
from python.similarity_engine import compute_similarity_pairs, build_similarity_matrix
from python.clustering import cluster_rhymes, get_threshold
from python.html_generation import filter_clusters

similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)
word_to_cluster, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

passing_labels = filter_clusters(word_to_cluster, rhyme_candidates)

passing = {}
for entry in rhyme_candidates:
    word = entry["end_word"]
    label = word_to_cluster.get(word)
    if label in passing_labels:
        passing.setdefault(label, [])
        if word not in passing[label]:
            passing[label].append(word)

print("Passing clusters:")
for label, members in sorted(passing.items()):
    print(f"  {label}: {members}")

Passing clusters:
  B: ['found', 'sound', 'underground', 'around', 'mound']
  D: ['of', 'was']
  E: ['crowd', 'loud', 'proud']
  G: ['light', 'night', 'kite', 'flight', 'sight', 'white', 'delight', 'bright', 'tonight']


---
## Section 7: HTML Generation <a id="html-generation"></a>
---

In [10]:
clusters = clusters = cluster_labels

generate_rhyme_html(rhyme_candidates, clusters, detection_mode=DETECTION_MODE, debug=True)

[HTML] Passing clusters: ['B', 'D', 'E', 'G']
[HTML] Blocked clusters: ['A', 'C', 'F', 'H', 'I', 'J', 'K']
HTML file generated: generated_html/rhyme_visualization.html
